In [19]:
!pip install triton

In [20]:
import torch
import torch.nn as nn
import numpy as np
from torch.fx import symbolic_trace
import math
import operator
import triton
import triton.language as tl
import torch.nn.functional as F
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import gc
# from torch.nn.attention import sdpa_kernel, SDPBackend

In [21]:
# Wrap the function in torch.compile to fuse the pointwise operations
@torch.compile
def quantize_to_int8(t: torch.Tensor):
    """
    Symmetrically quantizes a float tensor to INT8.
    """

    original_bytes = t.nelement() * t.element_size()

    # 127.0 because INT8 ranges from -128 to 127 (drop -128 for perfect symmetry)
    scale = t.abs().max() / 127.0

    # Divide by scale and round to nearest integer
    q_t = torch.round(t / scale)

    # Clamp to ensure no float small overflows issue, then strictly cast to int8.
    q_t = torch.clamp(q_t, -128, 127).to(torch.int8)

    new_bytes = q_t.nelement() * q_t.element_size()
    scale_bytes = scale.nelement() * scale.element_size()
    total_new_bytes = new_bytes + scale_bytes

    # Verify during Run Time
    # print(f"--- Quantization Intercept ---")
    # print(f"Original Tensor: {t.dtype} | {original_bytes} bytes")
    # print(f"Quantized Tensor: {q_t.dtype} | {new_bytes} bytes (Data) + {scale_bytes} bytes (Scale)")
    # print(f"Memory Reduction: {(1 - (total_new_bytes / original_bytes)) * 100:.2f}%")

    return q_t, scale

def dequantize_to_fp16(q_t: torch.Tensor, scale: torch.Tensor):
    """
    Dequantizes an INT8 tensor back to FP16 using the scale.
    """
    # Cast back to a float format before multiplying by the float scale
    return (q_t.to(torch.float16) * scale)


In [22]:
class NaiveMultiHeadAttentionBatched(nn.Module):
    def __init__(self, W, bias, Wo, bias_o, d_model, heads):
        super().__init__()
        self.d_model = d_model
        self.heads = heads
        self.d = d_model // heads

        self.W = nn.Parameter(W)
        self.bias = nn.Parameter(bias)
        self.Wo = nn.Parameter(Wo)
        self.bias_o = nn.Parameter(bias_o)

    def forward(self, X):
        # X is [Batch, SeqLen, d_model]
        B, N, _ = X.shape

        W_q = self.W[:, :self.d_model]
        W_k = self.W[:, self.d_model:2*self.d_model]
        W_v = self.W[:, 2*self.d_model:]

        b_q = self.bias[:self.d_model]
        b_k = self.bias[self.d_model:2*self.d_model]
        b_v = self.bias[2*self.d_model:]

        Q = torch.matmul(X, W_q) + b_q
        K = torch.matmul(X, W_k) + b_k
        V = torch.matmul(X, W_v) + b_v

        # Reshape and transpose into [Batch, Heads, SeqLen, HeadDim]
        Q = Q.view(B, N, self.heads, self.d).transpose(1, 2)
        K = K.view(B, N, self.heads, self.d).transpose(1, 2)
        V = V.view(B, N, self.heads, self.d).transpose(1, 2)

        # Batched Attention Math
        P = torch.matmul(Q, K.transpose(-2, -1))
        S = torch.softmax(P / math.sqrt(self.d), dim=-1)
        O = torch.matmul(S, V)

        # Transpose and flatten back to [Batch, SeqLen, d_model]
        O = O.transpose(1, 2).contiguous().view(B, N, self.d_model)

        output = torch.matmul(O, self.Wo) + self.bias_o
        return output

In [23]:
def transform(m: nn.Module) -> nn.Module:
    symbolic_traced = torch.fx.symbolic_trace(m)

    print(symbolic_traced.graph)
    # We use a set because in an unrolled multi-head loop,
    # we will find multiple distinct K and V nodes.
    nodes_to_quantize = set()

    for node in symbolic_traced.graph.nodes:
        # We will get K and V node based on known future use (where they are used as an argument)
        if node.target == torch.matmul:
            # torch.matmul(S, V)
            # If the first argument comes from a softmax, the second is V.
            arg0 = node.args[0]
            if isinstance(arg0, torch.fx.Node) and ("softmax" in str(arg0.target)):
                v_node = node.args[1]
                nodes_to_quantize.add(v_node)

            # torch.matmul(Q, K.T)
            # If the second argument is a transposed tensor, its input is K.
            arg1 = node.args[1]
            # .T in PyTorch usually traces as the string 't' or the function torch.transpose
            if isinstance(arg1, torch.fx.Node) and (str(arg1.target) == "t" or "transpose" in str(arg1.target)):
                # The input to the transpose operation is the raw K tensor
                k_node = arg1.args[0]
                nodes_to_quantize.add(k_node)

    print(f"Found {len(nodes_to_quantize)} K/V tensors to quantize.")

    # Injecting the Quantization logic
    for target_node in nodes_to_quantize:
        # Capture original users
        original_users = list(target_node.users.keys())

        # Inject the tuple node
        with symbolic_traced.graph.inserting_after(target_node):
            quantized_tuple_node = symbolic_traced.graph.call_function(
                quantize_to_int8, args=(target_node,)
            )

        # Extract the INT8 data (Anchor = tuple_node)
        with symbolic_traced.graph.inserting_after(quantized_tuple_node):
            q_t_node = symbolic_traced.graph.call_function(
                operator.getitem, args=(quantized_tuple_node, 0)
            )

        # Extract the FP16 scale (Anchor = q_t_node)
        with symbolic_traced.graph.inserting_after(q_t_node):
            scale_node = symbolic_traced.graph.call_function(
                operator.getitem, args=(quantized_tuple_node, 1)
            )

        # Dequantize (Anchor = scale_node)
        with symbolic_traced.graph.inserting_after(scale_node):
            dequantized_node = symbolic_traced.graph.call_function(
                dequantize_to_fp16, args=(q_t_node, scale_node)
            )

        # Safely reroute - we need to prevent cyclic dependency here
        for user_node in original_users:
            user_node.replace_input_with(target_node, dequantized_node)

    # validates the computational graph
    symbolic_traced.graph.lint()
    symbolic_traced.recompile()

    print(symbolic_traced.graph)

    return symbolic_traced

In [24]:
# An Simple implementation where just dequantization is fused to next stage to prevent FP16 write to HBM, but
# without any Flash ATtention, so it is still a O(N^2) operation

# Compile ONLY the QK phase.
# Inductor fuses the INT8->FP16 cast into the matmul prologue in SRAM.
# Because Softmax is missing, it CANNOT pattern-match into Flash Attention!
@torch.compile
def fused_qk_dequantize(q, k_int8, k_scale):
    k_fp16 = (k_int8.to(torch.float16) * k_scale).to(torch.float16)
    sm_scale = 1.0 / math.sqrt(q.shape[-1])
    # Forces the O(N^2) matrix to be written to HBM
    return torch.matmul(q, k_fp16.transpose(-2, -1)) * sm_scale

# 2. Compile ONLY the AV phase.
# Inductor fuses the INT8->FP16 cast into the matmul prologue in SRAM.
@torch.compile
def fused_av_dequantize(attn, v_int8, v_scale):
    v_fp16 = (v_int8.to(torch.float16) * v_scale).to(torch.float16)
    return torch.matmul(attn, v_fp16)

def true_unfused_int8_baseline(q, k_tuple, v_tuple):
    k_int8, k_scale = k_tuple
    v_int8, v_scale = v_tuple

    # Step A: Fused K dequantization -> Outputs O(N^2) matrix
    scores = fused_qk_dequantize(q, k_int8, k_scale)

    # Step B: Standard PyTorch Softmax (Reads O(N^2), Writes O(N^2))
    attn = torch.nn.functional.softmax(scores, dim=-1)

    # Step C: Fused V dequantization -> Outputs final projection
    out = fused_av_dequantize(attn, v_int8, v_scale)

    return out

In [25]:
torch.manual_seed(42)
B = 1
N = 128
d = 64
heads = 2
d_model = d * heads

# Move to CUDA for FP16 and Triton execution
X = torch.rand(B, N, d_model, dtype=torch.float16, device="cuda")

# 2. Initialize Models (Ensure batch_first=True for the PyTorch baseline)
mha = nn.MultiheadAttention(
    embed_dim=d_model, num_heads=heads, dtype=torch.float16, batch_first=True
).cuda()

naive_mha = NaiveMultiHeadAttentionBatched(
    W=mha.in_proj_weight.T,
    bias=mha.in_proj_bias,
    Wo=mha.out_proj.weight.T,
    bias_o=mha.out_proj.bias,
    d_model=d_model,
    heads=heads
).half().cuda()

# 3. Verify Unquantized Mathematical Equivalence
naive_out = naive_mha(X)
pytorch_out, _ = mha(X, X, X, need_weights=False)

# Using a slightly looser tolerance for FP16 math on GPU
is_match = torch.allclose(naive_out, pytorch_out, atol=1e-3)
print(f"Do the unquantized outputs match? {'YES!' if is_match else 'NO!'}")


# Tranform without Kernel Fusion
compiled_unfused_mha = transform(naive_mha)

Do the unquantized outputs match? YES!
graph():
    %x : [num_users=4] = placeholder[target=X]
    %getattr_1 : [num_users=3] = call_function[target=builtins.getattr](args = (%x, shape), kwargs = {})
    %getitem : [num_users=4] = call_function[target=operator.getitem](args = (%getattr_1, 0), kwargs = {})
    %getitem_1 : [num_users=4] = call_function[target=operator.getitem](args = (%getattr_1, 1), kwargs = {})
    %getitem_2 : [num_users=0] = call_function[target=operator.getitem](args = (%getattr_1, 2), kwargs = {})
    %w : [num_users=3] = get_attr[target=W]
    %getitem_3 : [num_users=1] = call_function[target=operator.getitem](args = (%w, (slice(None, None, None), slice(None, 128, None))), kwargs = {})
    %getitem_4 : [num_users=1] = call_function[target=operator.getitem](args = (%w, (slice(None, None, None), slice(128, 256, None))), kwargs = {})
    %getitem_5 : [num_users=1] = call_function[target=operator.getitem](args = (%w, (slice(None, None, None), slice(256, None, None)))

In [26]:
# Incorporating Flash attention to better compare with Pytorch Native MHA.

@triton.jit
def _attention(
        Q, K, V,
        O, l, m,
        stride_qh, stride_qn,
        stride_kh, stride_kn,
        stride_vh, stride_vn,
        stride_oh, stride_on,
        stride_lh,
        stride_mh,
        BLOCK_R : tl.constexpr,
        BLOCK_C : tl.constexpr,
        BLOCK_D : tl.constexpr,
        N_q : tl.constexpr, N_v : tl.constexpr,
        D_head : tl.constexpr, B_c : tl.constexpr, B_r : tl.constexpr,
        T_r : tl.constexpr, T_c : tl.constexpr,
        sm_scale,
        ):
    # program_id(0) = head index (parallelized over heads)
    # program_id(1) = Q block (row block) index (parallelized over T_r)
    # BLOCK_R == B_r and BLOCK_C == B_c for this implementation
    head_idx = tl.program_id(0)
    tr_idx = tl.program_id(1)

    # --- Base pointers per head ---
    Q_ptr = Q + head_idx * stride_qh
    K_ptr = K + head_idx * stride_kh
    V_ptr = V + head_idx * stride_vh
    O_ptr = O + head_idx * stride_oh
    l_ptr = l + head_idx * stride_lh
    m_ptr = m + head_idx * stride_mh

    # --- Offsets for the current Q block (rows) ---
    # Q block: rows [tr_idx * B_r, tr_idx * B_r + B_r)
    offs_q = tr_idx * BLOCK_R + tl.arange(0, BLOCK_R)       # [BLOCK_R]
    offs_d = tl.arange(0, BLOCK_D)                      # [BLOCK_D]

    # Load Q block: [BLOCK_R, BLOCK_D]
    # Load only once as it is common for this kernel
    qo_mask = (offs_q[:, None] < N_q) & (offs_d[None, :] < D_head)
    q_ptrs_2d = Q_ptr + offs_q[:, None] * stride_qn + offs_d[None, :]
    q = tl.load(q_ptrs_2d, mask=qo_mask, other=0.0)

    # Initialize running statistics
    m_i = tl.full([BLOCK_R], value=float('-inf'), dtype=tl.float16)  # row-wise max
    l_i = tl.zeros([BLOCK_R], dtype=tl.float32)                      # row-wise sum of exp
    o_i = tl.zeros([BLOCK_R, BLOCK_D], dtype=tl.float32)             # accumulator for output

    # --- Inner loop over K/V blocks (columns) ---
    for tc in range(T_c):

        offs_kv = tc * BLOCK_C + tl.arange(0, BLOCK_C)      # [BLOCK_C]
        kv_mask = (offs_kv[:, None] < N_v) & (offs_d[None, :] < D_head)

        # Load K block: [BLOCK_C, BLOCK_D]
        k_ptrs_2d = K_ptr + offs_kv[:, None] * stride_kn + offs_d[None, :]
        k = tl.load(k_ptrs_2d, mask=kv_mask, other=0.0)

        # Load V block: [BLOCK_C, BLOCK_D]
        v_ptrs_2d = V_ptr + offs_kv[:, None] * stride_vn + offs_d[None, :]
        v = tl.load(v_ptrs_2d, mask=kv_mask, other=0.0)

        # S = Q @ K^T
        s = tl.dot(q, k.T) * sm_scale                   # [BLOCK_R, BLOCK_C]

        # Mask out-of-bounds KV positions
        # Only required when N_v % B_c != 0 (-inf to zero out the exponent)
        s_mask = (offs_q[:, None] < N_q) & (offs_kv[None, :] < N_v)
        s = tl.where(s_mask, s, float('-inf'))

        # Row-wise max of current block
        m_ij = tl.max(s, axis=1)                        # [BLOCK_R]

        # New running max
        m_new = tl.maximum(m_i, m_ij)                   # [BLOCK_R]

        # Correction factor for previous accumulator
        alpha = tl.exp(m_i - m_new)
        # Softmax numerator for current block
        p = tl.exp(s - m_new[:, None])                  # [BLOCK_R, BLOCK_C]

        # Update running sum
        l_i = l_i * alpha + tl.sum(p, axis=1)           # [BLOCK_R]

        # Rescale previous output accumulator (step 10)
        o_i = o_i * alpha[:, None]                      # [BLOCK_R, BLOCK_D]

        # Accumulate: O += P @ V (step 10)
        o_i += tl.dot(p.to(v.dtype), v)                 # [BLOCK_R, BLOCK_D]

        # Update running max
        m_i = m_new.to(m_i.dtype)

    # --- Normalize output ---
    o_i = o_i / l_i[:, None]

    # --- Store outputs ---
    o_ptrs_2d = O_ptr + offs_q[:, None] * stride_on + offs_d[None, :]
    tl.store(o_ptrs_2d, o_i, mask=qo_mask)

    # Store l and m (per-row statistics, useful for backward pass)
    # q_mask_1d = offs_q < N_q
    # tl.store(l_ptr + offs_q, l_i, mask=q_mask_1d)
    # tl.store(m_ptr + offs_q, m_i, mask=q_mask_1d)


def multiheaded_attention_triton(
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        W_qkv, W_out,
        b_qkv, b_out,
        num_heads=1,
        device="cuda",
        block_size=64) -> torch.Tensor:

    N_q, D = query.shape
    N_v, _ = key.shape
    dtype = W_qkv.dtype
    D_head = D // num_heads

    # --- QKV Projections (full) ---
    Q = torch.matmul(query, W_qkv[0:D, :].T) + b_qkv[0:D][None, :]       # (N_q, D)
    K = torch.matmul(key, W_qkv[D:2*D, :].T) + b_qkv[D:2*D][None, :]     # (N_v, D)
    V = torch.matmul(value, W_qkv[2*D:3*D, :].T) + b_qkv[2*D:3*D][None, :] # (N_v, D)

    # --- Reshape to multi-head: (num_heads, seq_len, D_head) ---
    Q = Q.reshape(N_q, num_heads, D_head).permute(1, 0, 2).contiguous()  # (H, N_q, D_head)
    K = K.reshape(N_v, num_heads, D_head).permute(1, 0, 2).contiguous()  # (H, N_v, D_head)
    V = V.reshape(N_v, num_heads, D_head).permute(1, 0, 2).contiguous()  # (H, N_v, D_head)

    # --- Block sizes ---
    BLOCK_D = triton.next_power_of_2(D_head)
    B_c = min(triton.next_power_of_2(N_v), block_size)
    B_r = min(triton.next_power_of_2(N_q), block_size)
    BLOCK_C = B_c
    BLOCK_R = B_r

    T_c = triton.cdiv(N_v, B_c)  # number of KV blocks
    T_r = triton.cdiv(N_q, B_r)  # number of Q blocks

    # --- Allocate outputs ---
    O = torch.zeros_like(Q)                                               # (H, N_q, D_head)
    l = torch.zeros((num_heads, N_q), device=device, dtype=torch.float32)
    m = torch.full((num_heads, N_q), fill_value=float('-inf'), device=device, dtype=torch.float32)

    # --- Grid: parallelize over (heads, Q blocks) ---
    grid = (num_heads, T_r)
    if D_head <= block_size:
        num_warps = 4
    else:
        num_warps = 8

    sm_scale = 1.0 / math.sqrt(D_head)

    _attention[grid](
        Q, K, V,
        O, l, m,
        # Q strides: (H, N_q, D_head)
        Q.stride(0), Q.stride(1),
        # K strides: (H, N_v, D_head)
        K.stride(0), K.stride(1),
        # V strides: (H, N_v, D_head)
        V.stride(0), V.stride(1),
        # O strides: (H, N_q, D_head)
        O.stride(0), O.stride(1),
        # l strides: (H, N_q)
        l.stride(0),
        # m strides: (H, N_q)
        m.stride(0),
        BLOCK_R=BLOCK_R,
        BLOCK_C=BLOCK_C,
        BLOCK_D=BLOCK_D,
        N_q=N_q, N_v=N_v,
        D_head=D_head, B_c=B_c, B_r=B_r,
        T_r=T_r, T_c=T_c,
        sm_scale=sm_scale,
        num_warps=num_warps,
    )

    # --- Reshape output back: (H, N_q, D_head) -> (N_q, D) ---
    O = O.permute(1, 0, 2).contiguous().reshape(N_q, D)

    # --- Output projection ---
    out = torch.matmul(O, W_out.T) + b_out[None, :]

    return out

print("Flash Attention V2 kernel (_attention) and multiheaded_attention_triton defined.")


Flash Attention V2 kernel (_attention) and multiheaded_attention_triton defined.


In [27]:
@triton.autotune(
    configs=[
        triton.Config({'BLOCK_R': 64, 'BLOCK_C': 64}, num_warps=4, num_stages=3),
        triton.Config({'BLOCK_R': 128, 'BLOCK_C': 64}, num_warps=8, num_stages=4),
        triton.Config({'BLOCK_R': 64, 'BLOCK_C': 128}, num_warps=8, num_stages=4),
        triton.Config({'BLOCK_R': 128, 'BLOCK_C': 128}, num_warps=8, num_stages=5),
        triton.Config({'BLOCK_R': 64, 'BLOCK_C': 32}, num_warps=4, num_stages=5),
        triton.Config({'BLOCK_R': 32, 'BLOCK_C': 64}, num_warps=4, num_stages=5),
    ],
    key=['N_q', 'N_v', 'D_head'],
)
@triton.jit
def _fx_flash_attn_v2_int8kv(
    Q_ptr, K_int8_ptr, V_int8_ptr, O_ptr,
    k_scale_val, v_scale_val, sm_scale,
    stride_qb, stride_qh, stride_qn, stride_qd,
    stride_kb, stride_kh, stride_kn, stride_kd,
    stride_vb, stride_vh, stride_vn, stride_vd,
    stride_ob, stride_oh, stride_on, stride_od,
    N_q, N_v,
    D_head: tl.constexpr,
    BLOCK_R: tl.constexpr, BLOCK_C: tl.constexpr, BLOCK_D: tl.constexpr,
):
    batch_idx = tl.program_id(0)
    head_idx  = tl.program_id(1)
    tr_idx    = tl.program_id(2)

    # Base pointers
    Q = Q_ptr + batch_idx * stride_qb + head_idx * stride_qh
    K = K_int8_ptr + batch_idx * stride_kb + head_idx * stride_kh
    V = V_int8_ptr + batch_idx * stride_vb + head_idx * stride_vh
    O = O_ptr + batch_idx * stride_ob + head_idx * stride_oh

    offs_q = tr_idx * BLOCK_R + tl.arange(0, BLOCK_R)
    offs_d = tl.arange(0, BLOCK_D)

    # Load Q
    qo_mask = (offs_q[:, None] < N_q) & (offs_d[None, :] < D_head)
    q_ptrs = Q + offs_q[:, None] * stride_qn + offs_d[None, :] * stride_qd
    q = tl.load(q_ptrs, mask=qo_mask, other=0.0)

    # Initialize running statistics
    m_i = tl.full([BLOCK_R], float("-inf"), dtype=tl.float32)
    l_i = tl.zeros([BLOCK_R], dtype=tl.float32)
    o_i = tl.zeros([BLOCK_R, BLOCK_D], dtype=tl.float32)

    T_c = tl.cdiv(N_v, BLOCK_C)

    for tc in range(T_c):
        offs_kv = tc * BLOCK_C + tl.arange(0, BLOCK_C)

        # 1. Pre-transpose K load (Swap rows/cols to bypass the .T compiler bug)
        kv_mask_T = (offs_d[:, None] < D_head) & (offs_kv[None, :] < N_v)
        k_ptrs = K + offs_d[:, None] * stride_kd + offs_kv[None, :] * stride_kn
        k_int8 = tl.load(k_ptrs, mask=kv_mask_T, other=0)
        k = (k_int8.to(tl.float16) * k_scale_val).to(tl.float16)

        # 2. Standard linear load for V
        kv_mask = (offs_kv[:, None] < N_v) & (offs_d[None, :] < D_head)
        v_ptrs = V + offs_kv[:, None] * stride_vn + offs_d[None, :] * stride_vd
        v_int8 = tl.load(v_ptrs, mask=kv_mask, other=0)
        v = (v_int8.to(tl.float16) * v_scale_val).to(tl.float16)

        # 3. Compute dot product WITHOUT .T (K is already transposed in SRAM)
        s = tl.dot(q, k, out_dtype=tl.float32) * sm_scale

        s_mask = (offs_q[:, None] < N_q) & (offs_kv[None, :] < N_v)
        s = tl.where(s_mask, s, float("-inf"))

        m_ij  = tl.max(s, axis=1)
        m_new = tl.maximum(m_i, m_ij)

        alpha = tl.exp(m_i - m_new)
        p     = tl.exp(s - m_new[:, None])

        l_i = l_i * alpha + tl.sum(p, axis=1)
        o_i = o_i * alpha[:, None]
        o_i += tl.dot(p.to(tl.float16), v, out_dtype=tl.float32)
        m_i = m_new

    o_i = o_i / l_i[:, None]

    o_ptrs = O + offs_q[:, None] * stride_on + offs_d[None, :] * stride_od
    tl.store(o_ptrs, o_i.to(tl.float16), mask=qo_mask)

# The Python Wrapper for the FX Pass
def fx_flash_attn_v2_wrapper(q, k_tuple, v_tuple):
    # Unpack the tuples generated by the quantize_to_int8 node
    k_int8, k_scale = k_tuple
    v_int8, v_scale = v_tuple

    B, H, N_q, D_head = q.shape
    N_v = k_int8.shape[2]

    BLOCK_D = triton.next_power_of_2(D_head)
    O = torch.empty_like(q)
    sm_scale = 1.0 / math.sqrt(D_head)

    # Dynamic Grid based on Autotuned BLOCK_R
    grid = lambda META: (B, H, triton.cdiv(N_q, META['BLOCK_R']))

    _fx_flash_attn_v2_int8kv[grid](
        q, k_int8, v_int8, O,
        k_scale.item(), v_scale.item(), sm_scale,
        q.stride(0), q.stride(1), q.stride(2), q.stride(3),
        k_int8.stride(0), k_int8.stride(1), k_int8.stride(2), k_int8.stride(3),
        v_int8.stride(0), v_int8.stride(1), v_int8.stride(2), v_int8.stride(3),
        O.stride(0), O.stride(1), O.stride(2), O.stride(3),
        N_q, N_v,
        D_head=D_head,
        BLOCK_D=BLOCK_D
    )
    return O

In [28]:
def transform_to_triton_batched(m: nn.Module, use_fused_FA : bool = False) -> nn.Module:
    symbolic_traced = torch.fx.symbolic_trace(m)
    attention_outputs = []

    # 1. Traverse AST to map the attention subgraph
    for node in symbolic_traced.graph.nodes:
        if node.target == torch.matmul:
            arg0 = node.args[0]
            if isinstance(arg0, torch.fx.Node) and ("softmax" in str(arg0.target)):
                final_matmul = node
                softmax_node = arg0
                v_node = final_matmul.args[1]

                div_node = softmax_node.args[0]
                qk_matmul = div_node.args[0]

                q_node = qk_matmul.args[0]
                k_transpose_node = qk_matmul.args[1]
                k_node = k_transpose_node.args[0]

                attention_outputs.append({
                    "final_out": final_matmul,
                    "softmax": softmax_node,
                    "div_node": div_node,
                    "qk_matmul": qk_matmul,
                    "q": q_node,
                    "k": k_node,
                    "v": v_node,
                    "k_transpose": k_transpose_node
                })

    # 2. Graph Rewriting
    for att_block in attention_outputs:
        # Step A: Inject Quantization nodes
        with symbolic_traced.graph.inserting_before(att_block["qk_matmul"]):
            quantized_k = symbolic_traced.graph.call_function(quantize_to_int8, args=(att_block["k"],))
        with symbolic_traced.graph.inserting_before(att_block["qk_matmul"]):
            quantized_v = symbolic_traced.graph.call_function(quantize_to_int8, args=(att_block["v"],))


        attention_node = fx_flash_attn_v2_wrapper if use_fused_FA else true_unfused_int8_baseline

        # Step B: Inject Flash Attention Kernel
        with symbolic_traced.graph.inserting_after(quantized_v):
            triton_node = symbolic_traced.graph.call_function(
                attention_node,
                args=(att_block["q"], quantized_k, quantized_v)
            )

        # Step C: Reroute and Clean Up
        att_block["final_out"].replace_all_uses_with(triton_node)

        symbolic_traced.graph.erase_node(att_block["final_out"])
        symbolic_traced.graph.erase_node(att_block["softmax"])
        symbolic_traced.graph.erase_node(att_block["div_node"])
        symbolic_traced.graph.erase_node(att_block["qk_matmul"])
        symbolic_traced.graph.erase_node(att_block["k_transpose"])

    symbolic_traced.graph.lint()
    symbolic_traced.recompile()

    return symbolic_traced

In [32]:
def benchmark_attention(setting: str = "original"):
    """
    Modular attention benchmark.

    Parameters
    ----------
    setting : "original" | "flash_int8_kv"
        "original"      — Compares Native vs Naive vs FX-Compiled INT8
        "flash_int8_kv" — Compares Native vs Pure Triton Flash vs FX-Compiled INT8
    """
    assert setting in ("original", "flash_int8_kv"), (
        f"Unknown setting '{setting}'. Choose 'original' or 'flash_int8_kv'."
    )

    torch.manual_seed(42)
    d = 64
    heads = 4
    d_model = d * heads

    seq_lengths = [256, 512, 1024, 2048, 4096, 8192, 16384, 32768]

    results = []

    for N in seq_lengths:
        print(f"[{setting}] Benchmarking Sequence Length: {N}...")

        try:
            X = torch.rand(1, N, d_model, dtype=torch.float16, device="cuda")
        except RuntimeError:
            print(f"  -> Fatal OOM allocating input tensor at {N}. Stopping.")
            break

        pytorch_mha = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=heads, dtype=torch.float16, batch_first=True
        ).cuda()

        naive_mha = NaiveMultiHeadAttentionBatched(
            W=pytorch_mha.in_proj_weight.T, bias=pytorch_mha.in_proj_bias,
            Wo=pytorch_mha.out_proj.weight.T, bias_o=pytorch_mha.out_proj.bias,
            d_model=d_model, heads=heads
        ).half().cuda()

        compiled_mha = transform_to_triton_batched(naive_mha, use_fused_FA = False)
        compiled_FA_mha = transform_to_triton_batched(naive_mha, use_fused_FA = True)

        if setting == "original":
            models = {
                "PyTorch Native":        lambda: pytorch_mha(X, X, X, need_weights=False)[0],
                "Naive Unrolled":        lambda: naive_mha(X),
                "Triton Unfused (INT8 KV)": lambda: compiled_mha(X),
            }
        else:  # "flash_int8_kv"
            models = {
                "PyTorch Native":              lambda: pytorch_mha(X, X, X, need_weights=False)[0],
                "Flash Attn V2 (Triton)":      lambda: multiheaded_attention_triton(
                    X[0], X[0], X[0],
                    W_qkv=pytorch_mha.in_proj_weight,
                    W_out=pytorch_mha.out_proj.weight,
                    b_qkv=pytorch_mha.in_proj_bias,
                    b_out=pytorch_mha.out_proj.bias,
                    num_heads=heads, device="cuda", block_size=64,
                ),
                "Flash Attn V2 + INT8 KV (FX)": lambda: compiled_FA_mha(X),
            }

        seq_label = f"{N//1024}k" if N >= 1024 else str(N)

        for model_name, forward_fn in models.items():
            gc.collect()
            torch.cuda.empty_cache()

            try:
                with torch.inference_mode():# Warmup (Crucial for Triton Autotuning)
                  for _ in range(3):
                      _ = forward_fn()
                  torch.cuda.synchronize()

                  torch.cuda.reset_peak_memory_stats()


                  # Timed run
                  start_evt = torch.cuda.Event(enable_timing=True)
                  end_evt   = torch.cuda.Event(enable_timing=True)
                  start_evt.record()
                  for _ in range(10):
                      _ = forward_fn()
                  end_evt.record()
                  torch.cuda.synchronize()

                latency_ms   = start_evt.elapsed_time(end_evt) / 10.0
                peak_mem_mb  = torch.cuda.max_memory_allocated() / (1024 ** 2)

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"  -> [OOM] {model_name} at {seq_label}")
                    latency_ms, peak_mem_mb = None, None
                    torch.cuda.empty_cache()
                else:
                    raise

            results.append({
                "Seq_Val":         N,
                "Seq_Label":       seq_label,
                "Model":           model_name,
                "Latency (ms)":    latency_ms,
                "Peak Memory (MB)": peak_mem_mb,
                "Setting":         setting,
            })

    df = pd.DataFrame(results)

    if setting == "original":
        colors = {
            "PyTorch Native":         "#94a3b8",
            "Naive Unrolled":         "#ef4444",
            "Triton Unfused (INT8 KV)": "#0f172a",
        }
        chart_title = "Attention Architecture: FX Compilation vs O(N^2) Baseline"
    else:
        colors = {
            "PyTorch Native":               "#94a3b8",
            "Flash Attn V2 (Triton)":       "#10b981",
            "Flash Attn V2 + INT8 KV (FX)": "#6d28d9",
        }
        chart_title = "Attention Architecture: Compiled INT8 Flash vs Native Flash"


    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Forward Pass Latency (ms)", "Peak HBM Allocation (MB)"),
        horizontal_spacing=0.1,
    )

    for model_name, color in colors.items():
        model_data = df[df["Model"] == model_name]

        # Latency line
        fig.add_trace(
            go.Scatter(
                x=model_data["Seq_Label"],
                y=model_data["Latency (ms)"],
                mode="lines+markers",
                name=model_name,
                line=dict(color=color, width=2.5, shape="spline"),
                marker=dict(size=7, opacity=0.9),
                legendgroup=model_name,
                connectgaps=False,
            ),
            row=1, col=1,
        )

        # Memory bar
        fig.add_trace(
            go.Bar(
                x=model_data["Seq_Label"],
                y=model_data["Peak Memory (MB)"],
                name=model_name,
                marker_color=color,
                marker_line_width=0,
                legendgroup=model_name,
                showlegend=False,
            ),
            row=1, col=2,
        )

    axis_styling = dict(
        showgrid=True, gridwidth=1, gridcolor="#f1f5f9",
        showline=True, linewidth=1, linecolor="#e2e8f0",
        tickfont=dict(color="#64748b", size=12),
        title_font=dict(size=14, color="#475569"),
    )

    fig.update_layout(
        title=dict(
            text=chart_title,
            font=dict(size=22, color="#0f172a"), x=0.01, y=0.95,
        ),
        height=600, plot_bgcolor="#ffffff", paper_bgcolor="#ffffff",
        margin=dict(t=120, b=40, l=40, r=40),
        legend=dict(
            orientation="h", yanchor="bottom", y=1.05, xanchor="left", x=0,
            font=dict(size=13, color="#475569"), bgcolor="rgba(0,0,0,0)",
        ),
        hovermode="x unified",
        barmode="group",
        bargap=0.15, bargroupgap=0.05,
    )

    fig.update_xaxes(title_text="Sequence Length (Log2 Progression)", **axis_styling, row=1, col=1)
    fig.update_yaxes(title_text="Latency (ms) — Log Scale", type="log", **axis_styling, row=1, col=1)
    fig.update_xaxes(title_text="Sequence Length (Log2 Progression)", **axis_styling, row=1, col=2)
    fig.update_yaxes(title_text="Memory (MB)", **axis_styling, row=1, col=2)

    fig.show()
    return df

In [33]:
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

df_original = benchmark_attention(setting="original")
display(df_original.pivot(index="Seq_Label", columns="Model",
                           values=["Latency (ms)", "Peak Memory (MB)"]))


[original] Benchmarking Sequence Length: 256...
[original] Benchmarking Sequence Length: 512...
[original] Benchmarking Sequence Length: 1024...
[original] Benchmarking Sequence Length: 2048...
[original] Benchmarking Sequence Length: 4096...
[original] Benchmarking Sequence Length: 8192...
[original] Benchmarking Sequence Length: 16384...
[original] Benchmarking Sequence Length: 32768...


Latency (ms)                                          \
Model     Naive Unrolled PyTorch Native Triton Unfused (INT8 KV)   
Seq_Label                                                          
16k            11.343667       2.180915                10.880819   
1k              0.340890       0.254771                 0.978944   
256             0.390656       0.242995                 1.038541   
2k              0.396698       0.234189                 1.029939   
32k            41.046732       6.380647                42.533887   
4k              1.087488       0.277197                 1.246925   
512             0.383590       0.253440                 1.035571   
8k              4.215705       0.725811                 4.506112   

          Peak Memory (MB)                                          
Model       Naive Unrolled PyTorch Native Triton Unfused (INT8 KV)  
Seq_Label                                                           
16k            6187.164551      67.164551              4147.165527  
1k               37.164551      15.712402                29.665527  
256              13.164551      12.039551                12.790527  
2k              111.164551      22.290527                80.165527  
32k           24651.164551     123.164551             16475.165527  
4k              403.164551      25.164551               277.165527  
512              18.164551      13.438965                16.415527  
8k             1563.164551      39.164551              1055.165527

In [34]:
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

df_flash_int8 = benchmark_attention(setting="flash_int8_kv")
display(df_flash_int8.pivot(index="Seq_Label", columns="Model",
                             values=["Latency (ms)", "Peak Memory (MB)"]))


[flash_int8_kv] Benchmarking Sequence Length: 256...
[flash_int8_kv] Benchmarking Sequence Length: 512...
[flash_int8_kv] Benchmarking Sequence Length: 1024...
[flash_int8_kv] Benchmarking Sequence Length: 2048...
[flash_int8_kv] Benchmarking Sequence Length: 4096...
[flash_int8_kv] Benchmarking Sequence Length: 8192...
[flash_int8_kv] Benchmarking Sequence Length: 16384...
[flash_int8_kv] Benchmarking Sequence Length: 32768...


Latency (ms)                                              \
Model     Flash Attn V2 (Triton) Flash Attn V2 + INT8 KV (FX) PyTorch Native   
Seq_Label                                                                      
16k                     2.824909                     4.801536       2.179891   
1k                      0.496640                     0.795238       0.233472   
256                     0.569651                     0.839066       0.240128   
2k                      0.579379                     0.878490       0.236954   
32k                     7.899136                    12.967220       8.140903   
4k                      0.583578                     0.874906       0.276275   
512                     0.551731                     0.819814       0.256922   
8k                      0.863130                     1.505690       0.725811   

                Peak Memory (MB)                                              
Model     Flash Attn V2 (Triton) Flash Attn V2 + INT8 KV (FX) PyTorch Native  
Seq_Label                                                                     
16k                    67.664551                    51.164551      67.164551  
1k                     14.695801                    13.664551      15.712402  
256                    12.047363                    11.789551      12.039551  
2k                     18.227051                    16.164551      22.290527  
32k                   124.164551                    91.164551     123.164551  
4k                     25.289551                    21.164551      25.164551  
512                    12.930176                    12.414551      13.438965  
8k                     39.414551                    31.164551      39.164551